In [1]:
# Importing standard libraries
import os
import math
import numpy as np
import pandas as pd
import cv2
# Importing libraries from TensorFlow and Keras for deep learning
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    GlobalMaxPooling2D,
)
from tensorflow.keras.optimizers import RMSprop, Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
# Importing libraries for model building and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import confusion_matrix, roc_curve, auc
import warnings
warnings.simplefilter("always")

# Importing libraries for handling DICOM files and displaying progress
import pydicom as dcm
from glob import glob
from tqdm import notebook



In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adagrad,Adam


In [3]:
labels_dir = "labels/"
output_dir = "output/"
data_dir = "data/"
train_images_dir = "stage_2_train_images/"
test_images_dir = "stage_2_test_images/"

TRAIN_IMAGE_LABELS_FILE = os.path.join(labels_dir, "stage_2_train_labels.csv")
CLASS_INFO_FILE = os.path.join(labels_dir, "stage_2_detailed_class_info.csv")
TEST_IMAGE_LABELS_FILE = os.path.join(labels_dir, "stage_2_test_labels.csv")

In [4]:
# Read the data from the saved numpy files to avoid running the previous cells
PATIENT_IDS = np.load(data_dir + "patientIds.npy")
TARGETS = np.load(data_dir + "targets.npy")
TRAIN_IMAGES = np.load(data_dir + "trainImages.npy")

X = TRAIN_IMAGES

y = to_categorical(TARGETS)
NUM_CLASSES = 2

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

In [23]:
HEIGHT = 128
WIDTH = 128
NUM_CHANNELS = 3
NUM_CLASSES = 2

model3 = Sequential()
model3.add(Input(shape=(HEIGHT, WIDTH, NUM_CHANNELS)))

# Start with fewer filters and lower dropout rate
model3.add(Conv2D(filters=8, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.05))

model3.add(Conv2D(filters=16, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.05))

model3.add(Conv2D(filters=32, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.10))

model3.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.10))

model3.add(Conv2D(filters=128, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.15))

model3.add(GlobalMaxPooling2D())
model3.add(Dense(256, activation="relu"))
model3.add(Dropout(0.5))
model3.add(Dense(NUM_CLASSES, activation="softmax"))

model3.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model3.summary()


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_50 (Conv2D)              │ (None, 126, 126, 8)    │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_42 (MaxPooling2D) │ (None, 63, 63, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_53 (Dropout)            │ (None, 63, 63, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_51 (Conv2D)              │ (None, 61, 61, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_43 (MaxPooling2D) │ (None, 30, 30, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_54 (Dropout)            │ (None, 30, 30, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_52 (Conv2D)              │ (None, 28, 28, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_44 (MaxPooling2D) │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_55 (Dropout)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_53 (Conv2D)              │ (None, 12, 12, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_45 (MaxPooling2D) │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_56 (Dropout)            │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_54 (Conv2D)              │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_46 (MaxPooling2D) │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_57 (Dropout)            │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d_11         │ (None, 128)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_58 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 131,922 (515.32 KB)

 Trainable params: 131,922 (515.32 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
mcp_save = ModelCheckpoint(
    output_dir + "model3.keras", save_best_only=True, monitor="val_loss", mode="min"
)

history_model3 = model3.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=64,
    callbacks=[mcp_save],
)

Epoch 1/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 38s 95ms/step - accuracy: 0.6752 - loss: 1.8186 - val_accuracy: 0.7659 - val_loss: 0.5146
Epoch 2/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 35s 94ms/step - accuracy: 0.7613 - loss: 0.5003 - val_accuracy: 0.7746 - val_loss: 0.5102
Epoch 3/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 41s 107ms/step - accuracy: 0.7640 - loss: 0.4920 - val_accuracy: 0.7817 - val_loss: 0.4911
Epoch 4/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 43s 114ms/step - accuracy: 0.7720 - loss: 0.4806 - val_accuracy: 0.7787 - val_loss: 0.4962
Epoch 5/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 45s 120ms/step - accuracy: 0.7709 - loss: 0.4831 - val_accuracy: 0.7774 - val_loss: 0.4730
Epoch 6/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 40s 105ms/step - accuracy: 0.7783 - loss: 0.4735 - val_accuracy: 0.7847 - val_loss: 0.4863
Epoch 7/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 39s 102ms/step - accuracy: 0.7791 - loss: 0.4692 - val_accuracy: 0.7853 - val_loss: 0.4743
Epoch 8/10
378/378 ━━━━━━━━━━━━━━━━━━━━ 39s 103ms/step - accuracy: 0.7755 - loss: 0.4

In [1]:
import matplotlib.pyplot as plt

accuracy = history_model3.history["accuracy"]
val_accuracy = history_model3.history["val_accuracy"]
loss = history_model3.history["loss"]
val_loss = history_model3.history["val_loss"]
N = range(100)

plt.figure(figsize=(15, 15))
plt.subplot(2, 2, 1)
plt.plot(N, accuracy, label="Training Accuracy")
plt.plot(N, val_accuracy, label="Validation Accuracy")
plt.legend()
plt.title("Training and Validation Accuracy")

plt.subplot(2, 2, 2)
plt.plot(N, loss, label="Training Loss")
plt.plot(N, val_loss, label="Validation Loss")
plt.legend()
plt.title("Training and Validation Loss")
plt.show()

ImportError: DLL load failed while importing _imaging: The specified module could not be found.

In [ ]:
model3 = Sequential()
model3.add(Input(shape=(HEIGHT, WIDTH, NUM_CHANNELS)))

# Start with fewer filters and lower dropout rate
model3.add(Conv2D(filters=8, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.05))

model3.add(Conv2D(filters=16, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.05))

model3.add(Conv2D(filters=32, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.10))

model3.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.10))

model3.add(Conv2D(filters=128, kernel_size=(3, 3), activation="relu"))
model3.add(MaxPooling2D(pool_size=(2, 2)))
model3.add(Dropout(0.15))

# Add global pooling and dense layers
model3.add(GlobalMaxPooling2D())
model3.add(Dense(512, activation="relu"))  # Increased number of units
model3.add(Dropout(0.5))  # Dropout to prevent overfitting
model3.add(Dense(NUM_CLASSES, activation="softmax"))

model3.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model3.summary()